# Bangladesh Climate-Smart Farming: Crop Yield Prediction

**Models:** Random Forest Regressor vs XGBoost Regressor  
**Target:** crop `yield`  
**Dataset:** Bangladesh district-level crop and CRU crop-season climate data (1984–2023)

This notebook trains a separate model for each crop (Aus, Aman, Boro, Jute, Potato). This is important because the yield units differ: jute is measured in bales/acre while the other crops use tonnes/acre.

## Evaluation design
- Training period: 1984–2015
- Held-out temporal test period: 2016–2023
- Features: year, district, and six crop-season climate variables from CRU TS
- Metrics: MAE, RMSE, and R²

**Important:** This is a historical prediction benchmark. The held-out years are not used during model fitting. Results measure performance on later years in the supplied dataset, not guaranteed future farm-level accuracy.

## 1. Kaggle environment setup

1. Create a new Kaggle Notebook.
2. Click **Add Input** and upload `dataset_upload.zip` as a Kaggle Dataset (or add the extracted dataset folder).
3. Set the notebook accelerator to **CPU**. GPU is not required for these two tabular models.
4. Run all cells from top to bottom.

Internet is not required if the ZIP is already attached as notebook input. If XGBoost is not installed, the setup cell attempts to install it.

In [ ]:
# Install only if XGBoost is unavailable
import importlib.util, subprocess, sys
if importlib.util.find_spec("xgboost") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])

import os, glob, zipfile, warnings, json, joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
print("Python environment ready")
print("Pandas:", pd.__version__)

## 2. Locate and extract the uploaded dataset

This cell searches Kaggle's `/kaggle/input` directory for either the ZIP archive or already-extracted CSV files. If you run outside Kaggle, it also checks the current working directory.

In [ ]:
INPUT_ROOTS = [Path("/kaggle/input"), Path(".")]
zip_candidates = []
for root in INPUT_ROOTS:
    if root.exists():
        zip_candidates.extend(root.rglob("dataset_upload.zip"))

# Find extracted files first
long_candidates, cru_candidates = [], []
for root in INPUT_ROOTS:
    if root.exists():
        long_candidates.extend(root.rglob("crop_climate_long.csv"))
        cru_candidates.extend(root.rglob("cru_crop_season_climate.csv"))

if long_candidates and cru_candidates:
    LONG_PATH = long_candidates[0]
    CRU_PATH = cru_candidates[0]
    print("Using extracted CSV files:")
    print(LONG_PATH)
    print(CRU_PATH)
elif zip_candidates:
    EXTRACT_DIR = Path("/kaggle/working/agri_dataset") if Path("/kaggle/working").exists() else Path("./agri_dataset")
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_candidates[0], "r") as zf:
        zf.extractall(EXTRACT_DIR)
    LONG_PATH = next(EXTRACT_DIR.rglob("crop_climate_long.csv"))
    CRU_PATH = next(EXTRACT_DIR.rglob("cru_crop_season_climate.csv"))
    print("Extracted:", zip_candidates[0])
    print("Using:", LONG_PATH)
    print("Using:", CRU_PATH)
else:
    raise FileNotFoundError(
        "Dataset not found. In Kaggle, click Add Input and attach dataset_upload.zip "
        "or the extracted dataset folder. Expected crop_climate_long.csv and "
        "cru_crop_season_climate.csv."
    )

## 3. Load and merge crop yield with crop-season climate

We deliberately use CRU crop-season variables rather than the annual BMD station variables. The dataset documentation says BMD station values for 2014–2023 are withheld, while the CRU crop-season climate table has complete coverage for the supplied crop records.

In [ ]:
crop_df = pd.read_csv(LONG_PATH)
climate_df = pd.read_csv(CRU_PATH)

climate_features = [
    "tmax_mean_c",
    "tmin_mean_c",
    "rain_total_mm",
    "vapour_pressure_hpa",
    "cloud_mean_okta",
    "wet_days",
]
join_keys = ["district", "year", "crop"]

required_crop_cols = join_keys + ["yield"]
required_climate_cols = join_keys + climate_features

missing_crop = sorted(set(required_crop_cols) - set(crop_df.columns))
missing_climate = sorted(set(required_climate_cols) - set(climate_df.columns))
if missing_crop or missing_climate:
    raise ValueError(f"Missing columns. Crop file: {missing_crop}; climate file: {missing_climate}")

data = crop_df[required_crop_cols].merge(
    climate_df[required_climate_cols],
    on=join_keys,
    how="inner",
    validate="one_to_one"
)

data["year"] = pd.to_numeric(data["year"], errors="coerce")
data["yield"] = pd.to_numeric(data["yield"], errors="coerce")
for col in climate_features:
    data[col] = pd.to_numeric(data[col], errors="coerce")

data = data.dropna(subset=["district", "year", "crop", "yield"] + climate_features).copy()
data["year"] = data["year"].astype(int)
data["district"] = data["district"].astype(str)

print("Merged dataset shape:", data.shape)
print("Crop records:", data["crop"].value_counts().sort_index().to_dict())
print("Year range:", data["year"].min(), "to", data["year"].max())
print("Missing values after cleaning:", int(data.isna().sum().sum()))
display(data.head())

## 4. Exploratory data checks

Each crop is modeled separately because the target units are not identical across all crops. Jute yield is in bales/acre; the other crop yields are in tonnes/acre.

In [ ]:
display(data.groupby("crop")["yield"].agg(["count", "mean", "std", "min", "max"]).round(4))
plt.figure(figsize=(9, 4))
data.boxplot(column="yield", by="crop", grid=False)
plt.title("Yield distribution by crop")
plt.suptitle("")
plt.xlabel("Crop")
plt.ylabel("Yield (crop-specific unit per acre)")
plt.tight_layout()
plt.show()

## 5. Define the two models and preprocessing

`district` is categorical and is one-hot encoded. Numeric variables are passed to the tree models without scaling. Missing feature values, if any, are imputed using the median within the training pipeline.

We train one Random Forest and one XGBoost model **for each crop**, rather than mixing crops with different yield units.

In [ ]:
numeric_features = ["year"] + climate_features
categorical_features = ["district"]
feature_columns = numeric_features + categorical_features

def make_preprocessor():
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features)
    ])

def build_models():
    rf = Pipeline([
        ("preprocess", make_preprocessor()),
        ("model", RandomForestRegressor(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=2,
            max_features=1.0,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

    xgb = Pipeline([
        ("preprocess", make_preprocessor()),
        ("model", XGBRegressor(
            n_estimators=500,
            max_depth=4,
            learning_rate=0.03,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="reg:squarederror",
            reg_lambda=2.0,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
    return {"Random Forest": rf, "XGBoost": xgb}

print("Models configured: Random Forest Regressor and XGBoost Regressor")

## 6. Train and evaluate using a temporal holdout

The split is chronological:
- Train: 1984–2015
- Test: 2016–2023

This is more realistic than a random row split for a climate/yield forecasting task because later years are held out from training.

In [ ]:
TRAIN_END_YEAR = 2015
TEST_START_YEAR = 2016

results = []
trained_models = {}
test_predictions = []

for crop_name in sorted(data["crop"].unique()):
    crop_data = data[data["crop"] == crop_name].sort_values("year").copy()
    train = crop_data[crop_data["year"] <= TRAIN_END_YEAR].copy()
    test = crop_data[crop_data["year"] >= TEST_START_YEAR].copy()

    if train.empty or test.empty:
        print(f"Skipping {crop_name}: train or test partition is empty.")
        continue

    X_train, y_train = train[feature_columns], train["yield"]
    X_test, y_test = test[feature_columns], test["yield"]

    print(f"\n===== {crop_name} | train={len(train)} | test={len(test)} =====")
    for model_name, model in build_models().items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        mae = mean_absolute_error(y_test, pred)
        rmse = np.sqrt(mean_squared_error(y_test, pred))
        r2 = r2_score(y_test, pred)

        results.append({
            "crop": crop_name,
            "model": model_name,
            "train_rows": len(train),
            "test_rows": len(test),
            "train_period": f"1984-{TRAIN_END_YEAR}",
            "test_period": f"{TEST_START_YEAR}-2023",
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

        trained_models[(crop_name, model_name)] = model
        pred_frame = test[["district", "year", "crop", "yield"]].copy()
        pred_frame["model"] = model_name
        pred_frame["predicted_yield"] = pred
        test_predictions.append(pred_frame)

        print(f"{model_name:14s} | MAE={mae:.5f} | RMSE={rmse:.5f} | R2={r2:.4f}")

results_df = pd.DataFrame(results).sort_values(["crop", "RMSE"]).reset_index(drop=True)
predictions_df = pd.concat(test_predictions, ignore_index=True)

print("\nFinal model comparison:")
display(results_df.round(5))

## 7. Compare model performance visually

Lower MAE and RMSE indicate smaller prediction errors. Higher R² indicates that the model explains more variation in the held-out target values. Select the model based on the held-out results, not on training performance.

In [ ]:
for metric in ["RMSE", "MAE", "R2"]:
    pivot = results_df.pivot(index="crop", columns="model", values=metric)
    ax = pivot.plot(kind="bar", figsize=(9, 4), rot=0)
    ax.set_title(f"Random Forest vs XGBoost: {metric}")
    ax.set_ylabel(metric)
    ax.set_xlabel("Crop")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

best_models = (
    results_df.sort_values("RMSE")
    .groupby("crop", as_index=False)
    .first()[["crop", "model", "MAE", "RMSE", "R2"]]
)
print("Best model per crop (lowest temporal test RMSE):")
display(best_models.round(5))

## 8. Inspect actual vs predicted yield

The following plots show held-out test observations only.

In [ ]:
for crop_name in sorted(predictions_df["crop"].unique()):
    crop_pred = predictions_df[predictions_df["crop"] == crop_name]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, model_name in zip(axes, ["Random Forest", "XGBoost"]):
        part = crop_pred[crop_pred["model"] == model_name]
        ax.scatter(part["yield"], part["predicted_yield"], alpha=0.45)
        low = min(part["yield"].min(), part["predicted_yield"].min())
        high = max(part["yield"].max(), part["predicted_yield"].max())
        ax.plot([low, high], [low, high], linestyle="--")
        ax.set_title(f"{crop_name}: {model_name}")
        ax.set_xlabel("Actual yield")
        ax.set_ylabel("Predicted yield")
    plt.tight_layout()
    plt.show()

## 9. Save trained models and result files

Saved outputs will be available in Kaggle's `/kaggle/working/` directory and can be downloaded from the notebook's Output panel.

In [ ]:
OUTPUT_DIR = Path("/kaggle/working/agri_yield_outputs") if Path("/kaggle/working").exists() else Path("./agri_yield_outputs")
MODEL_DIR = OUTPUT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(OUTPUT_DIR / "model_comparison_results.csv", index=False)
predictions_df.to_csv(OUTPUT_DIR / "test_set_predictions.csv", index=False)
best_models.to_csv(OUTPUT_DIR / "best_model_per_crop.csv", index=False)

for (crop_name, model_name), fitted_model in trained_models.items():
    safe_model = model_name.lower().replace(" ", "_")
    joblib.dump(fitted_model, MODEL_DIR / f"{crop_name.lower()}_{safe_model}.joblib")

with open(OUTPUT_DIR / "experiment_config.json", "w") as f:
    json.dump({
        "target": "yield",
        "features": feature_columns,
        "climate_source": "CRU TS crop-season climate",
        "train_period": "1984-2015",
        "test_period": "2016-2023",
        "models": ["Random Forest Regressor", "XGBoost Regressor"],
        "random_state": RANDOM_STATE
    }, f, indent=2)

print("Saved outputs to:", OUTPUT_DIR)
print("Files:", [p.name for p in OUTPUT_DIR.iterdir()])

## 10. How to explain this experiment to judges

**Problem:** Crop yield in Bangladesh is affected by changing temperature, rainfall, humidity-related conditions and local geography.

**Dataset:** We used district-level yield records for five crops across 64 districts from 1984–2023, joined with crop-season climate indicators from CRU TS.

**Target:** Yield (the dataset's `yield` column). We trained separate models for each crop because the target units are not the same for jute and the other crops.

**Models:** Random Forest is a bagging-based ensemble of decision trees. XGBoost is a gradient-boosting ensemble that builds trees sequentially to reduce previous errors.

**Features:** Year, district, and six crop-season climate indicators: mean maximum temperature, mean minimum temperature, total seasonal rainfall, vapour pressure, mean cloud cover and wet days.

**Evaluation:** We trained on 1984–2015 and evaluated on 2016–2023. We reported MAE, RMSE and R². Lower MAE/RMSE is better; higher R² is better.

**Limitations to disclose:**
1. These are district-level historical estimates, not direct sensor measurements from an individual farm.
2. Historical model performance does not guarantee future yield accuracy under unprecedented climate conditions.
3. CRU TS is gridded/interpolated climate data, not a farm's own weather-station reading.
4. District and year are included as features; the model learns historical geographic and temporal patterns.
5. A proper future deployment should be validated with recent local field observations and monitored for data drift.

**Do not claim a model is accurate until you have run the notebook and reported its actual test metrics.**